In [1]:
from pymongo import MongoClient
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
from bson import ObjectId
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline, TFBertForSequenceClassification, BertTokenizer, create_optimizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
import tensorflow as tf

In [ ]:
EPOCHS = 2
BATCH_SIZE = 12
NUM_LABELS = 3
LEARNING_RATE = 1e-5
FOLDS = 5
VOLATILITY = 30
RSI = 14
ROC = 8
MODEL_NAME = "yiyanghkust/finbert-tone"
MARKET = 'BTC-USD'
STARTING_DATE = "2023-03-01"
ENDING_DATE = "2023-07-01"
TWEETS_ID = '67082fb60891532d63a3ed67'
TRANSFORMATIONS = ["REMOVE_URLS", "REMOVE_HEX", "REMOVE_EMOTICONS", "REMOVE_USERNAMES"]
HASHTAGS_TO_KEEP = ["btc"]
RANDOM_SAMPLE = True
DAILY_SAMPLE_SIZE = 2

In [2]:
def remove_unwanted_hashtags(text, keep):
    def replacer(match):
        hashtag = match.group(0)[1:]
        if hashtag.lower() in keep:
            return match.group(0)
        else:
            return ''
    cleaned =  re.sub(r'[@$#]\w+', replacer, text)
    return re.sub(r'\s{2,}', ' ', cleaned).strip()

def apply_transformations(transformations, text):
    if "REMOVE_USERNAMES" in transformations:
        remove_usernames_pattern = r'@\S+'
        text = re.sub(remove_usernames_pattern, "", text)
    if "REMOVE_URLS" in transformations:
        remove_urls_pattern = r'http\S+'
        text = re.sub(remove_urls_pattern, "", text)
    if "REMOVE_PUNCTUATION_MARKS" in transformations:
        remove_pun_pattern_1 = r"(?<!\d)\.(?!\d)|[^\w\s.']"
        remove_pun_pattern_2 = r"'"
        remove_pun_pattern_3 = r"\s\s+"
        sub1 = re.sub(remove_pun_pattern_1, " ", text)
        sub2 = re.sub(remove_pun_pattern_2, "", sub1)
        text = re.sub(remove_pun_pattern_3, " ", sub2)
    if "TEXT_TO_LOWER" in transformations:
        text = text.lower()
    if "REMOVE_SHORT_WORDS" in transformations:
        remove_short_pattern = r'\b\w{1,2}\b'
        text = re.sub(remove_short_pattern, '', text)
    if "REMOVE_ENDING_HASHTAGS" in transformations:
        pattern = r'([#$@\$]\w+)(?=(\s[#$@\$]\w+)*\s*$)'
        text = re.sub(pattern, '', text)
    if "REMOVE_HEX" in transformations:
        text = re.sub(r'\b0x[a-fA-F0-9]{6,}\b', '', text)
    if "REMOVE_EMOTICONS" in transformations:
        emoji_pattern = re.compile(
            "[" 
            u"\U0001F600-\U0001F64F"  # Emoticons
            u"\U0001F300-\U0001F5FF"  # Symbols & pictographs
            u"\U0001F680-\U0001F6FF"  # Transport & map
            u"\U0001F1E0-\U0001F1FF"  # Flags
            u"\U00002700-\U000027BF"  # Dingbats
            u"\U0001F900-\U0001F9FF"  # Supplemental Symbols & Pictographs (includes 🤝)
            u"\U00002600-\U000026FF"  # Misc symbols (e.g. ☀️☂️)
            u"\U00002B00-\U00002BFF"  # Arrows etc.
            u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
            "]+", flags=re.UNICODE
        )
        text = emoji_pattern.sub(r'', text)
    return re.sub(r"\s{2,}", " ", text).strip()
def rsi_to_label(rsi_series):
    return np.where(rsi_series < 30, "bullish", np.where(rsi_series > 70, "bearish", "neutral"))

In [ ]:
client = MongoClient('mongodb://localhost:27017/')  
print('Connected to MongoDB')
db = client['phd']
tweets = db['tweets']
daily_tweets = db['daily_tweets']

In [4]:
data = yf.download(MARKET, start=STARTING_DATE, end=ENDING_DATE).reset_index().rename(columns={"Date": "date"})
data.columns = data.columns.get_level_values(0)
data = data.rename(columns={"date": "Date"})
data["Date"] = pd.to_datetime(data["Date"])
data = data[["Date", "Close", "Open", "High", "Low"]]
data['LogReturn'] = np.log(data['Close'] / data['Close'].shift(1))
data['Volatility'] = data['LogReturn'].ewm(span=VOLATILITY).std()
data["RSI"] = ta.rsi(data["Close"], length=RSI)
data["RSI_label"] = rsi_to_label(data["RSI"])
data["ROC"] = data['Close'].pct_change(periods=ROC) * 100
roc_std = data['ROC'].rolling(window=ROC).std()
data['ROC_label'] = np.where(data['ROC'] > roc_std, 'bullish', np.where(data['ROC'] < -roc_std, 'bearish', 'neutral'))
data = data.dropna()
data["datetime"] = pd.to_datetime(data["Date"]).dt.date

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed

1 Failed download:
['BTC-USD']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


In [13]:
def calculate_barriers(price, volatility, Fu, Fl):
    upper = price + price * volatility * Fu
    lower = price - price * volatility * Fl
    return upper, lower

def assign_labels(df, Fu, Fl, Vt):
    labels = []
    for i in range(len(df) - Vt):
        window = df.iloc[i:i+Vt+1]
        price_start = window['Close'].iloc[0]
        vol = window['Volatility'].iloc[0]
        upper, lower = calculate_barriers(price_start, vol, Fu, Fl)
        price_max = window['Close'].max()
        price_min = window['Close'].min()
        if price_max >= upper:
            labels.append('bullish')
        elif price_min <= lower:
            labels.append('bearish')
        else:
            labels.append('neutral')
    labels.extend([None] * Vt)
    return labels

def simulate_returns(df, labels, Vt):
    returns = []
    for i in range(len(df) - Vt):
        start_price = df['Close'].iloc[i]
        end_price = df['Close'].iloc[i+Vt]
        if labels[i] == 'bullish':
            ret = (end_price - start_price) / start_price
        elif labels[i] == 'bearish':
            ret = (start_price - end_price) / start_price
        else:
            ret = 0
        returns.append(ret)
    returns.extend([0] * Vt)
    return np.array(returns)

def optimize_sharpe(df, Fu_range, Fl_range, Vt_range, rf=0.04):
    best_sr = -np.inf
    best_params = None
    for Fu in Fu_range:
        for Fl in Fl_range:
            for Vt in Vt_range:
                labels = assign_labels(df, Fu, Fl, Vt)
                rets = simulate_returns(df, labels, Vt)
                mean_ret = np.mean(rets) * 252
                std_ret = np.std(rets) * np.sqrt(252)
                if std_ret > 0:
                    sr = (mean_ret - rf) / std_ret
                    if sr > best_sr:
                        best_sr = sr
                        best_params = (Fu, Fl, Vt)
    return best_sr, best_params

# Run optimization
Fu_range = np.arange(0.5, 3.1, 0.5)
Fl_range = np.arange(0.5, 3.1, 0.5)
Vt_range = range(8, 16)

best_sr, best_combination = optimize_sharpe(data, Fu_range, Fl_range, Vt_range)
print("Best Sharpe Ratio:", best_sr)
print("Best Parameters (Fu, Fl, Vt):", best_combination)

Best Sharpe Ratio: 13.331428652227938
Best Parameters (Fu, Fl, Vt): (2.5, 1.5, 15)
Best Sharpe Ratio: 13.331428652227938
Best Parameters (Fu, Fl, Vt): (2.5, 1.5, 15)


In [14]:
data["label"] = assign_labels(data, best_combination[0], best_combination[1], best_combination[2])
data["previous_label"] = data.shift(1)["label"]

In [15]:
tweets_aggregated = pd.DataFrame(tweets.find({
    "tag_id": ObjectId(TWEETS_ID),
    "created_at": {
        "$gte": datetime.strptime(STARTING_DATE, "%Y-%m-%d"),
        "$lte": datetime.strptime(ENDING_DATE, "%Y-%m-%d").replace(hour=23, minute=59, second=59, microsecond=999999)
    }
}))
tweets_aggregated["content"] = tweets_aggregated["content"].apply(lambda x: apply_transformations(TRANSFORMATIONS, remove_unwanted_hashtags(x, HASHTAGS_TO_KEEP)))
tweets_aggregated['datetime'] = pd.to_datetime(tweets_aggregated['created_at'])
tweets_aggregated = tweets_aggregated.sort_values(by='datetime')
tweets_aggregated['datetime'] = tweets_aggregated['datetime'].dt.date
if RANDOM_SAMPLE == True:  
    tweets_aggregated = tweets_aggregated.groupby('datetime', group_keys=False).sample(n=DAILY_SAMPLE_SIZE, random_state=42)
merged = pd.merge(tweets_aggregated, data, on='datetime', suffixes=('_df1', '_df2'))
merged = merged.dropna()

In [ ]:
# Load FinBERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

label_encoder = LabelEncoder()
targets = label_encoder.fit_transform(merged['label'])  # Maps up/down/neutral to 0/1/2
# Create prompt-based inputs
def create_prompt(tweet, prev_label, roc, rsi):
    return f"Previous Label: {prev_label}, ROC: {roc}, RSI: {rsi}, Tweet: {tweet}"

prompts = [create_prompt(t, pl, r, rs) for t, pl, r, rs in zip(merged['content'], merged['previous_label'], merged['ROC_label'], merged['RSI_label'])]

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(prompts)):
    print(f"\n=== Fold {fold + 1} ===")
    train_prompts = [prompts[i] for i in train_idx]
    val_prompts = [prompts[i] for i in val_idx]
    train_labels = tf.convert_to_tensor(targets[train_idx])
    val_labels = tf.convert_to_tensor(targets[val_idx])
    # Tokenize
    train_enc = tokenizer(train_prompts, truncation=True, padding=True, max_length=128, return_tensors="tf")
    val_enc = tokenizer(val_prompts, truncation=True, padding=True, max_length=128, return_tensors="tf")
    # Prepare dataset
    train_dataset = tf.data.Dataset.from_tensor_slices((
        {
            "input_ids": train_enc["input_ids"],
            "attention_mask": train_enc["attention_mask"],
            "token_type_ids": train_enc.get("token_type_ids", tf.zeros_like(train_enc["input_ids"]))  # for compatibility
        },
        train_labels
    )).shuffle(100).batch(BATCH_SIZE)

    val_dataset = tf.data.Dataset.from_tensor_slices((
        {
            "input_ids": val_enc["input_ids"],
            "attention_mask": val_enc["attention_mask"],
            "token_type_ids": val_enc.get("token_type_ids", tf.zeros_like(val_enc["input_ids"]))  # for compatibility
        },
        val_labels
    )).batch(BATCH_SIZE)
    
    model = TFBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
    for layer in model.bert.encoder.layer[:11]:
        layer.trainable = False 

    # # Optimizer & learning rate scheduler
    num_train_steps = len(merged) * EPOCHS
    num_warmup_steps = int(0.1 * num_train_steps)

    optimizer, lr_schedule = create_optimizer(
        init_lr=LEARNING_RATE,
        num_train_steps=num_train_steps,
        num_warmup_steps=num_warmup_steps
    )

    # Compile model
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )

    # Train
    model.fit(train_dataset,validation_data=val_dataset, epochs=2)

print("Training complete.")

C:\Users\micha\anaconda32\lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\micha\anaconda32\lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



=== Fold 1 ===

=== Fold 1 ===


C:\Users\micha\anaconda32\lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\micha\anaconda32\lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some layers from the model checkpoint at yiyanghkust/finbert-tone were not used when initializing TFBertForSequenceClassification: ['dropout_37']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you 

Epoch 1/2
Epoch 1/2
28286/28286 [==============================] - 9667s 341ms/step - loss: 0.8583 - accuracy: 0.7574 - val_loss: 3.1854 - val_accuracy: 0.3850
Epoch 2/2
Epoch 2/2
13002/28286 [============>.................] - ETA: 1:21:02 - loss: 0.4901 - accuracy: 0.8358